In [195]:
from underthesea import word_tokenize
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer
from sklearn.metrics import f1_score
import random
import torch
import json
import re
import pandas as pd
import numpy as np

In [196]:
PRIORITY = {"MACHINE": 1, "COMPONENT": 1, "ERROR_CODE": 1, "LOCATION": 1, "DEFECT_TYPE": 2}

entity_types = ['MACHINE','COMPONENT','DEFECT_TYPE','ERROR_CODE','LOCATION']

labels = [f'{k}{x}'for k in ['B-', 'I-'] for x in entity_types]
labels.insert(0, 'O')
id2string = {k: v for k, v in enumerate(labels)}
string2id = {v: k for k, v in enumerate(labels)}

In [197]:
raw_data = pd.read_csv('../data/sentences_labeled.csv')

In [198]:
entity_spans = []

for idx, row in raw_data.iterrows():
    tagged = []

    for i in entity_types:

        if pd.isna(row[i]): continue

        instances = [v.strip() for v in row[i].split(';')]

        sentence = row['sentence']

        start = 0

        for instance in instances:
            index = sentence.find(instance, start)

            index = sentence.find(instance, 0) if index < 0 else index

            start = index + len(instance)

            tagged.append((index, index + len(instance), i))

    entity_spans.append(tagged)


In [199]:
print(entity_spans)

[[(48, 67, 'MACHINE'), (0, 24, 'COMPONENT'), (43, 46, 'ERROR_CODE')], [(55, 79, 'LOCATION')], [(37, 44, 'MACHINE'), (0, 25, 'COMPONENT')], [(57, 65, 'COMPONENT'), (66, 70, 'DEFECT_TYPE')], [(44, 63, 'COMPONENT')], [(19, 43, 'MACHINE'), (48, 69, 'LOCATION')], [(15, 26, 'COMPONENT'), (27, 43, 'DEFECT_TYPE')], [(13, 17, 'COMPONENT'), (138, 151, 'COMPONENT'), (155, 158, 'DEFECT_TYPE')], [(0, 8, 'COMPONENT'), (13, 15, 'ERROR_CODE')], [(31, 43, 'MACHINE'), (60, 91, 'COMPONENT'), (15, 27, 'LOCATION')], [(45, 48, 'ERROR_CODE')], [(33, 46, 'COMPONENT'), (50, 64, 'DEFECT_TYPE')], [(4, 21, 'COMPONENT'), (33, 38, 'COMPONENT'), (39, 54, 'DEFECT_TYPE'), (58, 61, 'ERROR_CODE')], [(77, 85, 'MACHINE'), (47, 65, 'COMPONENT'), (99, 101, 'ERROR_CODE'), (19, 42, 'LOCATION')], [(19, 27, 'MACHINE'), (45, 50, 'COMPONENT'), (100, 108, 'COMPONENT')], [(0, 40, 'DEFECT_TYPE'), (44, 46, 'ERROR_CODE')], [(28, 41, 'LOCATION')], [(15, 37, 'MACHINE'), (63, 65, 'DEFECT_TYPE')], [(0, 14, 'COMPONENT'), (23, 31, 'COMPONEN

In [200]:
word_spans = []

for idx, row in raw_data.iterrows():
    sentence_data = []

    sentence = row['sentence']

    words = [v for v in word_tokenize(sentence)]
    
    start = 0

    for word in words:
        cleaned_word = word.replace("_", " ")

        index = sentence.find(cleaned_word, start)

        index = sentence.find(cleaned_word, 0) if index < 0 else index

        start = index + len(word)

        sentence_data.append((index, index + len(word), word))

    word_spans.append(sentence_data)


In [201]:
print(word_spans)

[[(0, 4, 'Biến'), (5, 19, 'tần Mitsubishi'), (20, 24, 'E700'), (25, 34, 'đột nhiên'), (35, 38, 'báo'), (39, 42, 'lỗi'), (43, 46, 'OC1'), (46, 47, ','), (48, 58, 'dây chuyền'), (59, 67, 'đóng gói'), (68, 72, 'đứng'), (73, 75, 'im'), (76, 79, 'rồi')], [(0, 5, 'Tháng'), (6, 12, '3/2024'), (12, 13, ','), (14, 18, 'mình'), (19, 23, 'nhận'), (24, 28, 'được'), (29, 37, 'cuộc gọi'), (38, 40, 'từ'), (41, 44, 'anh'), (45, 48, 'Hải'), (49, 50, '-'), (51, 54, 'chủ'), (55, 60, 'xưởng'), (61, 64, 'dệt'), (65, 68, 'vải'), (69, 70, 'ở'), (71, 79, 'Nam Định')], [(0, 4, 'Biến'), (5, 14, 'tần Delta'), (15, 20, 'VFD-B'), (21, 23, '15'), (23, 25, 'kW'), (26, 36, 'điều khiển'), (37, 40, 'máy'), (41, 44, 'dệt'), (45, 50, 'chính'), (51, 60, 'đột nhiên'), (61, 62, '"'), (62, 66, 'chết'), (67, 69, 'im'), (69, 70, '"')], [(0, 7, 'Ban đầu'), (8, 11, 'anh'), (12, 15, 'Hải'), (16, 19, 'nhờ'), (20, 23, 'thợ'), (24, 28, 'điện'), (29, 39, 'địa phương'), (40, 43, 'xem'), (43, 44, ','), (45, 53, 'kết luận'), (54, 56, 'l

In [202]:
claimed_list = []

for i in range(len(entity_spans)):
    sorted_row = sorted(entity_spans[i], key=lambda s: (PRIORITY[s[2]], s[0]))

    words = word_spans[i]

    tags = ['O' for i in range(len(words))]

    for span in sorted_row:
        last_claimed_idx = None
        
        for j in range(len(words)):
            word = words[j]

            if word[0] < span[1] and word[1] > span[0]:
                if tags[j] == "O":
                    if last_claimed_idx is None or word[0] != last_claimed_idx + 1:
                        tags[j] = f"B-{span[2]}" 
                        last_claimed_idx = word[1]
                    elif word[0] == last_claimed_idx + 1:
                        tags[j] = f"I-{span[2]}" 
                        last_claimed_idx = word[1]

    claimed_list.append(tags)        


In [203]:
print(claimed_list)

[['B-COMPONENT', 'I-COMPONENT', 'I-COMPONENT', 'O', 'O', 'O', 'B-ERROR_CODE', 'O', 'B-MACHINE', 'I-MACHINE', 'O', 'O', 'O'], ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-LOCATION', 'I-LOCATION', 'I-LOCATION', 'I-LOCATION', 'I-LOCATION'], ['B-COMPONENT', 'I-COMPONENT', 'I-COMPONENT', 'I-COMPONENT', 'B-COMPONENT', 'O', 'B-MACHINE', 'I-MACHINE', 'O', 'O', 'O', 'O', 'O', 'O'], ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-COMPONENT', 'B-DEFECT_TYPE', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O'], ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-COMPONENT', 'I-COMPONENT', 'O', 'O', 'O', 'O', 'O', 'O', 'O'], ['O', 'O', 'O', 'O', 'O', 'B-MACHINE', 'I-MACHINE', 'I-MACHINE', 'B-MACHINE', 'O', 'B-LOCATION', 'I-LOCATION', 'I-LOCATION', 'I-LOCATION'], ['O', 'O', 'O', 'B-COMPONENT', 'I-COMPONENT', 'B-DEFECT_TYPE', 'I-DEFECT_TYPE', 'I-DEFECT_TYPE', 'I-DEFECT_TYPE', 'O', 'O', 'O', 'O', 'O', 'O'], ['O', 'O', 'O', 'B-COMPONENT', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O'

In [204]:
tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base", use_fast=True)

In [205]:
processed_data = []

for i in range(len(word_spans)):
    words = [w[2] for w in word_spans[i]]
    tags = claimed_list[i]

    input_ids = [tokenizer.cls_token_id]
    labels = [-100]

    for word, tag in zip(words, tags):
        subword_ids = tokenizer.convert_tokens_to_ids(tokenizer.tokenize(word))
        input_ids.extend(subword_ids)
        labels.append(string2id[tag])
        labels.extend([-100] * (len(subword_ids) - 1))

    input_ids.append(tokenizer.sep_token_id)
    labels.append(-100)

    processed_data.append({"input_ids": input_ids, "labels": labels})

In [206]:
random.seed(36)
random.shuffle(processed_data)

n = len(processed_data)
n_train = int(n * 0.8)
n_val = int(n * 0.1)

train_data = processed_data[:n_train]
val_data = processed_data[n_train:n_train+n_val]
test_data = processed_data[n_train+n_val:]

splits = {"train": train_data, "val": val_data, "test": test_data}

for name, split in splits.items():
    with open(f"{name}.jsonl", "w", encoding="utf-8") as f:
        for row in split:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

with open("label_list.json", "w", encoding="utf-8") as f:
    json.dump({"string2id": string2id, "id2string": id2string}, f, ensure_ascii=False, indent=2)